# Phase 1.1: AIGCodeSet audit, cleaning and labeling


## 1. Load both files

In [1]:
import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

human = pd.read_csv("human_selected_dataset.csv")
ai    = pd.read_csv("created_dataset_with_llms.csv")

print("human:", human.shape)
print("ai:   ", ai.shape)

human: (4755, 16)
ai:    (2828, 6)


## 2. Data profile (task step 1)
What columns exist and what the categories look like. 
Findings so far: the human file has 16 columns (all Python, 1,585 samples per status). The AI file has only 6 columns. Its `status_in_folder` column tells you the scenario: `Generate` = from scratch, `Runtime` = fixing a runtime-error program, `Wrong` = fixing a wrong-answer program.

In [2]:
print("HUMAN columns:", human.columns.tolist())
print("AI columns:   ", ai.columns.tolist())
print()
print(human["status"].value_counts(), "\n")
print(human["original_language"].value_counts(), "\n")
print(ai["LLM"].value_counts(), "\n")
print(ai["status_in_folder"].value_counts(), "\n")
print("problems in human:", human["problem_id"].nunique(), " in ai:", ai["problem_id"].nunique())
print("missing values human:", int(human[["code","problem_id","submission_id"]].isna().sum().sum()),
      " ai:", int(ai.isna().sum().sum()))

HUMAN columns: ['submission_id', 'problem_id', 'user_id', 'date', 'language', 'original_language', 'filename_ext', 'status', 'cpu_time', 'memory', 'code_size', 'accuracy', 'status_in_folder', 'code', 'label', 'LLM']
AI columns:    ['problem_id', 'submission_id', 'LLM', 'status_in_folder', 'code', 'label']

status
Accepted         1585
Runtime Error    1585
Wrong Answer     1585
Name: count, dtype: int64 

original_language
Python (3.4.3)    2754
Python (3.8.2)     958
PyPy3 (2.4.0)      607
PyPy3 (7.3.0)      360
Python (2.7.6)      59
PyPy2 (5.6.0)       10
PyPy2 (7.3.0)        6
Python (2.7.3)       1
Name: count, dtype: int64 

LLM
GEMINI       948
LLAMA        942
CODESTRAL    938
Name: count, dtype: int64 

status_in_folder
Wrong       946
Runtime     941
Generate    941
Name: count, dtype: int64 

problems in human: 317  in ai: 317
missing values human: 0  ai: 0


## 3. Map both files to the team schema
Renames and recodes everything into the shared format: `sample_id, source, submission_id, problem_id, language, code_raw, label, llm_name, scenario, status, n_lines, n_chars`.
- `label`: 0 = human, 1 = AI
- `scenario`: human / from_scratch / fix_runtime_error / fix_wrong_answer
- `status`: for human code, its own status. For AI code, the status of the human code it was fixing (`none` for from-scratch)

In [3]:
h = pd.DataFrame({
    "submission_id": human["submission_id"],
    "problem_id":    human["problem_id"],
    "code_raw":      human["code"],
    "label":         0,
    "llm_name":      "human",
    "scenario":      "human",
    "status":        human["status"],
})

scenario_map = {"Generate": "from_scratch", "Runtime": "fix_runtime_error", "Wrong": "fix_wrong_answer"}
status_map   = {"Generate": "none",         "Runtime": "Runtime Error",     "Wrong": "Wrong Answer"}
llm_map      = {"GEMINI": "gemini", "LLAMA": "codellama", "CODESTRAL": "codestral"}

a = pd.DataFrame({
    "submission_id": ai["submission_id"],
    "problem_id":    ai["problem_id"],
    "code_raw":      ai["code"],
    "label":         1,
    "llm_name":      ai["LLM"].map(llm_map),
    "scenario":      ai["status_in_folder"].map(scenario_map),
    "status":        ai["status_in_folder"].map(status_map),
})

assert a["llm_name"].notna().all() and a["scenario"].notna().all()   # no unmapped values

df = pd.concat([h, a], ignore_index=True)
df["source"]   = "aigcodeset"
df["language"] = "python"

log = {"start_rows": len(df), "start_human": int((df.label==0).sum()), "start_ai": int((df.label==1).sum())}
print(log)
df.head(3)

{'start_rows': 7583, 'start_human': 4755, 'start_ai': 2828}


,submission_id,problem_id,code_raw,label,llm_name,scenario,status,source,language
0,s242112093,p03166,import sys\nsys.setrecursionlimit(1000000)\ndef dfs(u):\n if(dp[u]!=-1):\n return dp[u]\n for i in d[u]...,0,human,human,Accepted,aigcodeset,python
1,s683590509,p02686,print 'No',0,human,human,Runtime Error,aigcodeset,python
2,s750755997,p03147,"def solve(lst):\n def rec(left, right):\n if left == right:\n return 0\n res = 0\n ...",0,human,human,Runtime Error,aigcodeset,python


## 4. Basic cleaning (task step 2)
Drops missing/empty code and fixes line endings and invisible characters (we only touch these, nothing else, so the code is still the original).

In [4]:
df = df.dropna(subset=["code_raw"])
log["after_drop_nulls"] = len(df)

df["code_raw"] = (df["code_raw"].astype(str)
                  .str.replace("\r\n", "\n", regex=False)
                  .str.replace("\r", "\n", regex=False)
                  .str.replace("\ufeff", "", regex=False)
                  .str.replace("\x00", "", regex=False))

df = df[df["code_raw"].str.strip() != ""]
log["after_drop_empty"] = len(df)
print(log)

{'start_rows': 7583, 'start_human': 4755, 'start_ai': 2828, 'after_drop_nulls': 7583, 'after_drop_empty': 7583}


## 5. Same code under both labels(CONFLICT)
**Finding:** some AI outputs are identical to a human sample (the model returned the human code unchanged). The same text can't be both label 0 and label 1, so we remove the AI copies and keep the human ones.

In [5]:
human_codes = set(df.loc[df.label == 0, "code_raw"])
same_as_human = (df["label"] == 1) & df["code_raw"].isin(human_codes)
log["ai_identical_to_human_removed"] = int(same_as_human.sum())
print(df[same_as_human]["scenario"].value_counts())
df = df[~same_as_human]
log["after_conflicts"] = len(df)

scenario
fix_wrong_answer     65
fix_runtime_error    37
from_scratch          1
Name: count, dtype: int64


## 6. Exact duplicates

In [6]:
before = len(df)
dup = df.duplicated(subset=["code_raw"], keep="first")
log["dup_removed_human"] = int((dup & (df.label == 0)).sum())
log["dup_removed_ai"]    = int((dup & (df.label == 1)).sum())
df = df[~dup]
log["after_duplicates"] = len(df)
print(before, "->", len(df))

7480 -> 7453


## 7. Hunt for AI leftovers (task step 3)
Finds samples with markdown fences (```) or chat text such as "Here is the fixed code". The human column is the false-alarm baseline (it should be 0).

In [7]:
fence = df["code_raw"].str.contains("```", regex=False)
chat  = df["code_raw"].str.contains(
    r"(?im)^\s*(?:here is|here's|here are|sure[,!]|certainly|of course|"
    r"the (?:fixed|corrected|updated) code|explanation:|note:)", regex=True)

print(pd.DataFrame({"label": df["label"], "fence": fence, "chat": chat}).groupby("label").sum())
print()
print(df[(fence | chat) & (df.label == 1)]["llm_name"].value_counts())

for c in df[(fence | chat) & (df.label == 1)]["code_raw"].head(4):
    print("-" * 60)
    print(c[:300])

       fence  chat
label             
0          0     0
1         19    20

llm_name
codestral    21
codellama     8
Name: count, dtype: int64
------------------------------------------------------------
else:
    print(f)
Here is the fixed code: a, b, c = map(int, input().split())
d = a + b
e = a + c
f = b + c
print(min(d, e, f))
------------------------------------------------------------
n = int(input())
b = list(map(int, input().split()))
c = 0
d = 0
while c <= n - 3:
    if (b[c + 1] > b[c] and b[c + 1] < b[c + 2]) or (b[c + 1] < b[c] and b[c + 1] > b[c + 2]):
        d += 1
        c += 1
    else:
        c += 1

print(d)
```
The error in the code is that the `while` loop is not
------------------------------------------------------------
# Read the input string
direction_input = input()

# Initialize variables to keep track of the number of times Snuke travels in each direction
north_south_count = 0
east_west_count = 0

# Iterate through each character in the input string
for 

**Finding:** the flagged ones are real leftovers, for example "Here is the fixed code:" appearing in the middle of the file, so the code is not valid Python. We drop them, since keeping them would teach the model to detect chat text instead of coding style.

In [8]:
bad = (df["label"] == 1) & (fence | chat)
log["ai_leftovers_removed"] = int(bad.sum())
df = df[~bad]
log["after_leftovers"] = len(df)
print(log["ai_leftovers_removed"], "removed")

29 removed


## 8. Length check (task step 4)
Adds `n_chars` and `n_lines` and compares AI vs human.
**Finding:** the median lengths are close, so there is no strong "longer = AI" shortcut. But Gemini is clearly shorter, and about 100 human samples are under 50 characters. We flag these for later, and keep them in the file.

In [9]:
df["n_chars"] = df["code_raw"].str.len()
df["n_lines"] = df["code_raw"].str.count("\n") + 1

print(df.groupby("label")[["n_chars", "n_lines"]].describe().T)
print()
print(df.groupby("llm_name")["n_chars"].median())
print(df.groupby("scenario")["n_chars"].median())
log["human_under_50_chars"] = int(((df.label == 0) & (df.n_chars < 50)).sum())
log["ai_under_50_chars"]    = int(((df.label == 1) & (df.n_chars < 50)).sum())
print(log["human_under_50_chars"], log["ai_under_50_chars"])

label                     0            1
n_chars count   4750.000000  2674.000000
        mean     464.461895   417.858265
        std      647.984617   386.681063
        min        1.000000    12.000000
        25%      151.000000   159.000000
        50%      302.000000   304.000000
        75%      573.500000   545.750000
        max    19752.000000  5170.000000
n_lines count   4750.000000  2674.000000
        mean      23.044842    19.169409
        std       25.062648    15.431956
        min        1.000000     1.000000
        25%        9.000000     9.000000
        50%       17.000000    15.000000
        75%       29.000000    25.000000
        max      571.000000   222.000000

llm_name
codellama    316.0
codestral    357.0
gemini       254.5
human        302.0
Name: n_chars, dtype: float64
scenario
fix_runtime_error    294.5
fix_wrong_answer     293.5
from_scratch         317.0
human                302.0
Name: n_chars, dtype: float64
114 41


## 9. Save (task steps 5 and 6)
The scenario is already a column. Gives every row an ID, orders the columns, checks them, and writes the parquet file.

In [10]:
df = df.reset_index(drop=True)
df["sample_id"] = ["aigcodeset_" + str(i).zfill(6) for i in range(len(df))]

cols = ["sample_id", "source", "submission_id", "problem_id", "language", "code_raw",
        "label", "llm_name", "scenario", "status", "n_lines", "n_chars"]
df = df[cols]

assert df["sample_id"].is_unique
assert df["code_raw"].notna().all() and df["problem_id"].notna().all()
assert set(df["label"].unique()) == {0, 1}

log["final_rows"]  = len(df)
log["final_human"] = int((df.label == 0).sum())
log["final_ai"]    = int((df.label == 1).sum())

df.to_parquet("aigcodeset_clean.parquet", index=False)
pd.Series(log).to_csv("cleaning_log.csv", header=["count"])

print(pd.Series(log))
print()
print(df.groupby(["label", "llm_name", "scenario"]).size())

start_rows                       7583
start_human                      4755
start_ai                         2828
after_drop_nulls                 7583
after_drop_empty                 7583
ai_identical_to_human_removed     103
after_conflicts                  7480
dup_removed_human                   5
dup_removed_ai                     22
after_duplicates                 7453
ai_leftovers_removed               29
after_leftovers                  7424
human_under_50_chars              114
ai_under_50_chars                  41
final_rows                       7424
final_human                      4750
final_ai                         2674
dtype: int64

label  llm_name   scenario         
0      human      human                4750
1      codellama  fix_runtime_error     269
                  fix_wrong_answer      264
                  from_scratch          312
       codestral  fix_runtime_error     303
                  fix_wrong_answer      301
                  from_scratch          